In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))
import pandas as pd
import matplotlib
from src.metrics import MetricsCalculator

In [3]:
# load data
student_df = pd.read_csv("../data/processed/student_level.csv")
student_df.head()

,student_id,total_days,avg_engagement,engagement_variability,avg_difficulty,completion,satisfaction,learning,has_disability,disability_type,country,year,education_level,attendance_rate,early_engagement,engagement_trend
0,2024_0,12,3.533333,0.877893,2.916667,1.0,4.82,4.68,0,none,Kenya,2024,phd,0.800000,4.083333,-0.57
1,2024_10,9,3.475556,0.757663,3.000000,1.0,4.08,1.74,0,none,Peru,2024,masters,0.600000,3.323333,0.00
2,2024_1003,10,3.535000,1.102736,3.300000,1.0,3.14,4.31,0,none,Kenya,2024,phd,0.666667,4.086667,0.22
3,2024_1005,5,3.316000,0.400600,2.800000,0.0,3.26,3.11,0,none,UK,2024,undergrad,0.333333,3.350000,0.19
4,2024_1006,2,3.055000,0.671751,3.000000,0.0,1.84,4.09,1,multiple,USA,2024,masters,0.133333,3.055000,0.95


In [4]:
# initialise metrics calculator
metrics = MetricsCalculator(student_df)

In [5]:
# core metrics
completion_rate = metrics.completion_rate()
engagement_index = metrics.engagement_index()
learning_gain = metrics.learning_gain()
avg_satisfaction = metrics.avg_satisfaction()

print("Completion Rate:", round(completion_rate, 3))
print("Engagement Index:", round(engagement_index, 3))
print("Learning Gain:", round(learning_gain, 3))
print("Avg Satisfaction:", round(avg_satisfaction, 3))

Completion Rate: 0.448
Engagement Index: 0.681
Learning Gain: 1.816
Avg Satisfaction: 3.365


In [6]:
# risk metrics
# dropout rate
dropout_rate = 1 - completion_rate
print("Dropout Rate:", round(dropout_rate, 3))
# engagement var
engagement_std = student_df["avg_engagement"].std()
print("Engagement Variability:", round(engagement_std, 3))

Dropout Rate: 0.552
Engagement Variability: 0.448


insights high variability → inconsistent experience
	•	high dropout → program issue


#### Equity metrics

In [7]:
# disability equity gap
completion_by_disability = student_df.groupby("has_disability")["completion"].mean()
disability_gap = completion_by_disability.max() - completion_by_disability.min()

print("Disability Completion Gap:", round(disability_gap, 3))

Disability Completion Gap: 0.001


In [8]:
# by disability type
student_df.groupby("disability_type")["completion"].mean().sort_values()

disability_type
prefer_not_to_say    0.355556
motor                0.393939
multiple             0.435484
none                 0.448304
visual               0.464286
cognitive            0.468354
hearing              0.511905
Name: completion, dtype: float64

answer if 
“Are we serving all students equally?”


In [9]:
# country equity gap
completion_by_country = student_df.groupby("country")["completion"].mean()
country_gap = completion_by_country.max() - completion_by_country.min()

print("Country Equity Gap:", round(country_gap, 3))

Country Equity Gap: 0.112


#### Cohort Comparison

In [10]:
# metrics by year
results = []

for year, group in student_df.groupby("year"):
    m = MetricsCalculator(group)
    
    results.append({
        "year": year,
        "completion_rate": m.completion_rate(),
        "engagement_index": m.engagement_index(),
        "learning_gain": m.learning_gain(),
        "satisfaction": m.avg_satisfaction()
    })

metrics_by_year = pd.DataFrame(results)

metrics_by_year

,year,completion_rate,engagement_index,learning_gain,satisfaction
0,2024,0.452500,0.678363,1.845742,3.387750
1,2025,0.443908,0.684099,1.786524,3.343088


In [11]:
# comparing years 
metrics_by_year.set_index("year").plot(kind="bar")

ImportError: matplotlib is required for plotting when the default backend "matplotlib" is selected.

we need to Ensure consistency across years Is the program improving or declining?


#### Metric Validation

In [12]:
# range checks
assert 0 <= completion_rate <= 1
assert 0 <= engagement_index <= 1
assert 1 <= avg_satisfaction <= 5

In [13]:
# distribution check
student_df["completion"].value_counts(normalize=True)

student_df[[
    "attendance_rate",
    "avg_engagement",
    "completion"
]].corr()

,attendance_rate,avg_engagement,completion
attendance_rate,1.000000,0.027098,0.799359
avg_engagement,0.027098,1.000000,0.007283
completion,0.799359,0.007283,1.000000


Metrics should align logically:
	•	engagement ↑ → completion ↑


metric Definitions

In [14]:
# metric dict
METRIC_DEFINITIONS = {
    "completion_rate": "Proportion of students who completed the program",
    "engagement_index": "Average engagement normalized to [0,1]",
    "learning_gain": "Self-reported learning weighted by attendance",
    "dropout_rate": "Proportion of students who did not complete",
    "equity_gap": "Difference between highest and lowest group outcomes"
}

Group level metrics

In [15]:
# summ table
summary = {
    "Completion Rate": completion_rate,
    "Engagement Index": engagement_index,
    "Learning Gain": learning_gain,
    "Satisfaction": avg_satisfaction,
    "Dropout Rate": dropout_rate,
    "Disability Gap": disability_gap,
    "Country Gap": country_gap
}

pd.DataFrame.from_dict(summary, orient="index", columns=["Value"])

,Value
Completion Rate,0.448128
Engagement Index,0.681282
Learning Gain,1.815605
Satisfaction,3.365021
Dropout Rate,0.551872
Disability Gap,0.000685
Country Gap,0.111553


#### Final insights

Key Metrics Interpretation
	-	Completion rate indicates overall program success
	-	Engagement index suggests moderate student involvement
	-	Learning gain aligns strongly with attendance
	-	Disability gap highlights potential accessibility challenges
	-	Country gap suggests uneven outcomes across regions


Recommendations
	-	Improve early engagement strategies
	-	Provide targeted accessibility support
	-	Investigate regional disparities
